## Learning objectives
- Download public data from Zenodo
- apply AI (micro-sam) segmentation on an image

In [ ]:
#imports
import requests
import zipfile
from pathlib import Path
import io

In [ ]:
# get some data from zenodo


# URL of the file to download
url = "https://zenodo.org/records/14832406/files/sensitive_training_data.zip?download=1"

# Download the file
response = requests.get(url, stream=True)
response.raise_for_status()  # Raise an error for bad status codes

# Save the zip file temporarily
zip_path = Path("sensitive_training_data.zip")
with open(zip_path, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

In [ ]:
#unpack the zip file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("extracted_data")  # Extract to a folder named "extracted_data"

# Optional: Remove the zip file after extraction
zip_path.unlink()

In [ ]:
import os 
# list the folder structure
for root, dirs, files in os.walk("extracted_data"):
    level = root.replace("extracted_data", "").count(os.sep)
    indent = " " * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 4 * (level + 1)
    print(f"number of files: {len(files)}")

In [ ]:
import matplotlib
import numpy as np
from skimage.io import imread

from matplotlib import pyplot as plt

plt.figure(figsize=(20, 6))

img_path = "extracted_data/resistant_training_input/CR5_d0_1_Image_25917.tif"
img: np.ndarray = imread(img_path).astype(np.uint16)

plt.subplot(1, 2, 1)
plt.title("Input image")
plt.imshow(img, cmap='grey')
plt.axis("off")

label_img_path = "extracted_data/resistant_training_labels/CR5_d0_1_Image_25917.tif"
label_img: np.ndarray = imread(label_img_path).astype(np.uint16)

plt.subplot(1, 2, 2)
plt.title("Label image")
plt.imshow(label_img, cmap='grey')
plt.axis("off")

In [ ]:
#use default micro_sam model
from micro_sam.automatic_segmentation import get_predictor_and_segmenter, automatic_instance_segmentation

# vit_b_lm = ViT-Base finetuned on light microscopy (downloaded and cached on first use)
predictor, segmenter = get_predictor_and_segmenter(model_type="vit_b_lm")

segmentation = automatic_instance_segmentation(
    predictor=predictor,
    segmenter=segmenter,
    input_path=img,
    ndim=2,
)

print(f"number of objects found: {segmentation.max()}")

In [ ]:
plt.figure(figsize=(20, 6))

img_path = "extracted_data/resistant_training_input/CR5_d0_1_Image_25917.tif"
img: np.ndarray = imread(img_path).astype(np.uint16)

plt.subplot(1, 3, 1)
plt.title("Input image")
plt.imshow(img, cmap='grey')
plt.axis("off")

label_img_path = "extracted_data/resistant_training_labels/CR5_d0_1_Image_25917.tif"
label_img: np.ndarray = imread(label_img_path).astype(np.uint16)

plt.subplot(1, 3, 2)
plt.title("Label image")
plt.imshow(label_img, cmap='grey')
plt.axis("off")

plt.subplot(1, 3, 3)
plt.title("Micro-sam")
plt.imshow(segmentation, cmap='grey', alpha=0.5)


# Exercise

- apply on a few images and save the output

## Open image with napari and use micro_sam on it

In [ ]:
#import and open napari
import napari

viewer = napari.Viewer()
viewer.add_image(img, name="input", colormap="gray")
viewer.add_labels(label_img.astype("uint32"), name="ground truth")
viewer.add_labels(segmentation.astype("uint32"), name="micro-sam")

In [ ]:
# interactive micro_sam annotation in napari
# click points or draw a box on a cell, then press "s" to segment it
from micro_sam.sam_annotator import annotator_2d

# return_viewer=True keeps the notebook responsive (otherwise napari.run() blocks the cell)
annotator_viewer = annotator_2d(
    img,
    model_type="vit_b_lm",
    embedding_path="embeddings.zarr",  # cached, so re-running is fast
    return_viewer=True,
)

In [ ]:
## directly open the micro_sam plugin for annotation